# 05 – Nevralt nettverk, evaluering og forklaring

**Tid:** 180 minutter. 30 minutter repetisjon og modell, 45 minutter trening
og evaluering, 45 minutter feilanalyse, 45 minutter selvstendig oppsummering,
15 minutter felles avslutning. Sammenligning av modellstørrelser er valgfritt.

**Mål:** trene et nettverk med ett skjult lag, forklare tensorformene og
undersøke feil på nye bilder. Vi bruker CPU og ingen konvolusjonslag.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Eksempel: et skjult lag og ReLU

I notatbok 04 brukte vi ett lineært lag fra 784 piksler til ti skårer.
Nå legger vi inn et skjult lag med ReLU før det siste lineære laget.
Treningsløkken fra 04 kan brukes også med denne modellen.

Et skjult lag lager mellomliggende representasjoner. ReLU erstatter negative
verdier med 0 og beholder positive. Uten en ikke-lineær aktivering ville flere
lineære lag samlet fortsatt bare vært én lineær transformasjon.

Her er et lite eksempel med tre innganger, fire skjulte nevroner og to utganger.
Det siste laget gir rå skårer; vi legger ikke ReLU eller softmax etter det.


In [ ]:
small_model = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))
print(small_model(torch.ones(5, 3)).shape)
values = torch.tensor([-2., -1., 0., 1., 2.])
print("Før ReLU:", values, "Etter:", nn.ReLU()(values))


## Oppgave 1: Lag MNIST-nettverket

Skriv funksjonen `create_model(hidden_size)` som returnerer:
`Flatten → Linear(784, hidden_size) → ReLU → Linear(hidden_size, 10)`.
Opprett `model = create_model(128)`. Vis formen etter hvert lag for en gruppe
på 32 nullbilder, og tell modellens parametere.

<details>
<summary>Hint</summary>

Bruk en vanlig funksjon som returnerer `nn.Sequential`. Iterer over lagene for å vise mellomformene.

</details>

Før du bygger modellen: regn antall parametere som
`784 * H + H + H * 10 + 10`. Kontroller med koden for `H=128`.


In [ ]:
def create_model(hidden_size):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, 10),
    )

model = create_model(128)
values = torch.zeros(32, 1, 28, 28)
print("Inndata:", values.shape)
for layer in model:
    values = layer(values)
    print(type(layer).__name__, values.shape)
print("Parametere:", sum(parameter.numel() for parameter in model.parameters()))


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert model(torch.zeros(32, 1, 28, 28)).shape == (32, 10)

assert sum(p.numel() for p in model.parameters()) == 101770
assert model(torch.zeros(1, 1, 28, 28)).shape == (1, 10)


### Refleksjon

Skriv formene gjennom modellen med 128 skjulte nevroner.
Hva tilfører ReLU? Hvorfor har siste lag ti utganger, uansett størrelsen på det skjulte laget?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

`[32, 1, 28, 28] → [32, 784] → [32, 128] → [32, 128] → [32, 10]`.
ReLU tilfører ikke-linearitet uten å endre formen. Den lar modellen lære
andre sammenhenger enn en samlet lineær transformasjon. De ti utgangene
tilsvarer de ti mulige sifrene, ikke antall skjulte nevroner.


## Oppgave 2: Gjenbruk og forstå treningsløkken

Gjør treningskoden din fra notatbok 04 om til funksjonen
`train_model(model, train_loader, num_epochs=5, learning_rate=0.1)`.

Funksjonen skal opprette kryssentropitap og en ny SGD-optimizer, trene modellen
og returnere en liste med gjennomsnittlig treningstap per epoke. Antall
treningsbilder finnes med `len(train_loader.dataset)`.

Behold de fem treningsstegene synlige. Skriv ut tapet etter hver epoke.
Vi definerer funksjonen nå og bruker den til å trene nettverket under.

<details>
<summary>Hint</summary>

Flytt løkken inn i en funksjon. Opprett optimizer inne i funksjonen slik at den tilhører modellen som sendes inn.

</details>

Arbeid først i par: én forklarer rekkefølgen, én skriver. Bytt roller i evalueringen.
Hvis du står fast etter ti minutter, bruk løsningen til oppgave 2 som gjenoppretting
og fortsett med å forklare hva hver linje gjør.


In [ ]:
def train_model(model, train_loader, num_epochs=5, learning_rate=0.1):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(num_epochs):
        model.train()
        loss_sum = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            scores = model(images)
            loss = loss_fn(scores, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(images)
        history.append(loss_sum / len(train_loader.dataset))
        print(f"Epoke {epoch + 1}: treningstap {history[-1]:.4f}")
    return history


## Oppgave 3: Samle evalueringsresultater

Skriv `evaluate_model(model, data_loader)` ved å tilpasse testløkken fra
notatbok 04. Returner en ordbok med:

- `accuracy`: antall riktige delt på antall bilder.
- `predictions`: alle predikerte sifre, samlet i én tensor.
- `labels`: alle riktige sifre, samlet i samme rekkefølge.

Bruk `eval()` og `no_grad()`. Funksjonen skal ikke oppdatere modellen.

<details>
<summary>Hint</summary>

Samle tensorer i to lister og bruk `torch.cat` til slutt. Bruk `argmax(dim=1)` for prediksjoner.

</details>


In [ ]:
def evaluate_model(model, data_loader):
    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            scores = model(images)
            all_predictions.append(scores.argmax(dim=1))
            all_labels.append(labels)
    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    accuracy = (predictions == labels).float().mean().item()
    return {"accuracy": accuracy, "predictions": predictions, "labels": labels}


## Tren ett nettverk før du eksperimenterer

Vi bruker **128 skjulte nevroner**, batchstørrelse 64, fem epoker og læringsrate 0.1.
Hold disse innstillingene faste gjennom hovedløpet. Vi bruker `seed_everything(42)`
for å kunne gjenta forsøket. Den setter frø for Python, NumPy og PyTorch.

Testsettet brukes til avtalt rapportering og feilanalyse. Ikke velg nye
innstillinger ut fra testresultatene. Tuning krever et separat valideringssett,
som vi bare forklarer i dette introduksjonskurset. En valgfri sammenligning
av to forhåndsvalgte størrelser ligger etter hovedoppgavene.


In [ ]:
from time import perf_counter
from torch.utils.data import DataLoader
from pytorch_101.utils import seed_everything

# Eksempel på tidtaking; bruk samme mønster rundt selve treningen.
start = perf_counter()
sum(range(10_000))
print(f"Eksempelberegningen tok {perf_counter() - start:.6f} sekunder")


## Oppgave 4: Tren og rapporter nettverket

Opprett ordbøkene `models`, `histories`, `results` og `training_times`.
Bruk nøkkelen `128` for resultatene fra nettverket med 128 skjulte nevroner.

1. Sett frøet med `seed_everything(42)` og opprett en ny modell.
2. Opprett trenings- og testlastere med batchstørrelse 64. Stokk bare treningen.
   Gi treningslasteren `generator=torch.Generator().manual_seed(42)`.
3. Mål treningstiden, tren med `train_model` og evaluer med `evaluate_model`.
4. Lagre modellen, tapshistorikken, tiden og evalueringsresultatet i ordbøkene.
   Skriv ut sluttap, nøyaktighet og tid. Tegn treningskurven med aksetekster.

<details><summary>Hint</summary>

Bruk `perf_counter()` før og etter treningen. Ordbøkene gjør det mulig å legge
til en annen modell i den valgfrie oppgaven uten å endre resten av analysen.

</details>


In [ ]:
seed_everything(42)
model = create_model(128)
train_loader = DataLoader(
    train_data, batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
start = perf_counter()
histories = {128: train_model(model, train_loader)}
training_times = {128: perf_counter() - start}
results = {128: evaluate_model(model, test_loader)}
models = {128: model}
print(f"Sluttap: {histories[128][-1]:.4f}; "
      f"testnøyaktighet: {results[128]['accuracy']:.1%}; "
      f"treningstid: {training_times[128]:.2f} s")
fig, ax = plt.subplots()
ax.plot(range(1, 6), histories[128], marker="o")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="Nettverk med 128 skjulte nevroner")
plt.show()


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert 128 in models
for size in [128]:
    assert len(histories[size]) == 5
    assert np.isfinite(histories[size]).all()
    assert histories[size][-1] < histories[size][0]
    assert results[size]["predictions"].shape == (1000,)
    assert 0 <= results[size]["accuracy"] <= 1


### Refleksjon

Oppgi første og siste treningstap, testnøyaktighet og tid fra din kjøring.
Hva kan vi konkludere med? Kan vi avgjøre om modellen overtilpasser bare ved å
se at treningstapet synker? Hvorfor bør vi ikke velge læringsrate ut fra testresultatet?


**Mitt svar:**

Koden over skriver ut mine målte tall. Fallende treningstap viser at modellen
tilpasser seg treningsdata etter tapsfunksjonens mål. Testnøyaktighet måler
riktige klassifikasjoner på det separate utvalget. Treningstap alene kan ikke
avsløre overtilpasning; under utvikling ville vi sammenlignet med et valideringssett.
Hvis vi velger læringsrate ut fra testresultater, påvirker testdata modellvalget
og blir mindre egnet til en uavhengig sluttevaluering.


## Eksempel: finn posisjoner som oppfyller en betingelse

`torch.where` kan finne posisjonene der to tensorer er forskjellige.
Dette bruker vi til å finne feilklassifiserte bilder.


In [ ]:
example_predictions = torch.tensor([1, 2, 3, 4])
example_targets = torch.tensor([1, 7, 3, 9])
error_positions = torch.where(example_predictions != example_targets)[0]
print("Feil på posisjonene:", error_positions.tolist())


## Oppgave 5: Vis modellens feil

Bruk den forhåndsvalgte modellen med **128 skjulte nevroner**.
Finn feilindeksene fra `results[128]` og vis opptil ni feilklassifiserte
testbilder. Skriv riktig og predikert siffer i tittelen for hvert bilde.
Håndter også at modellen kan ha færre enn ni feil: skjul ubrukte akser.

`test_loader` stokker ikke, så resultatene har samme rekkefølge som `test_data`.

<details>
<summary>Hint</summary>

Kombiner `torch.where` med rutenettet fra notatbok 02. Bruk `index.item()` når du indekserer datasettet.

</details>


In [ ]:
predictions = results[128]["predictions"]
labels = results[128]["labels"]
error_indices = torch.where(predictions != labels)[0][:9]
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax in axes.ravel():
    ax.axis("off")
for ax, index in zip(axes.ravel(), error_indices):
    position = index.item()
    image, label = test_data[position]
    ax.imshow(image.squeeze(0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Riktig: {label}, gjettet: {predictions[position].item()}")
fig.suptitle("Eksempler på feilklassifiseringer")
fig.tight_layout()
plt.show()
print("Antall feil i hele testutvalget:", (predictions != labels).sum().item())


### Refleksjon

Beskriv to konkrete feil fra figuren. Ser noen sifre like ut?
Er ni utvalgte feil nok til å si hvilken klasse modellen er dårligst på?
Hva måtte vi undersøkt for å svare bedre på det?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et mulig eksempel er at en 5-er tolkes som 3 fordi deler av strekene ligner,
eller at en rund 6-er tolkes som 0. Beskriv to feil fra din egen figur;
bildene kan variere. Visuell likhet er en mulig forklaring, ikke et bevis på
hvilke trekk modellen brukte. De ni første feilene viser ikke hvilken klasse
som er vanskeligst: vi må telle riktige og feil per klasse i hele utvalget.


## Ferdig eksempel: vis sannsynligheter for ett bilde

Softmax omformer skårene til positive tall som summerer til 1.
Dette er modellens beregnede sannsynligheter, ikke en garanti for at den har rett.
Vi bruker softmax til visning, aldri før kryssentropitapet.


In [ ]:
image, label = test_data[0]
models[128].eval()
with torch.no_grad():
    scores = models[128](image.unsqueeze(0))
    probabilities = scores.softmax(dim=1).squeeze(0)
fig, ax = plt.subplots()
ax.bar(range(10), probabilities.numpy())
ax.set_xticks(range(10))
ax.set(xlabel="Siffer", ylabel="Beregnet sannsynlighet", title=f"Riktig siffer: {label}")
plt.show()


## Selvstendig oppsummering – 45 minutter

Her skal du sette sammen delene med mindre støtte. Du kan slå opp tidligere
eksempler, men skriv koden og forklaringene selv. Arbeid gjerne to og to,
med hver deres skriftlige svar.


## Oppgave 6: Bygg og forklar hele kjeden

Lag en **ny** modell med `Flatten`, `Linear(784, 64)`, `ReLU` og `Linear(64, 10)`.
Kall den `own_model`. Opprett nye datalastere direkte med `DataLoader` (64 bilder per gruppe,
trening med stokking og test uten stokking) og
en ny SGD-optimizer med læringsrate 0.1. Tren modellen i én epoke og evaluer
den på testutvalget. Skriv både treningsløkken og evalueringsløkken direkte
i cellen; ikke kall `train_model` eller `evaluate_model`.

Skriv ut gjennomsnittlig treningstap og testnøyaktighet. Denne korte repetisjonen
er ikke en rettferdig ytelsessammenligning med modellene som fikk fem epoker.

<details>
<summary>Hint</summary>

Skriv først rekkefølgen med egne ord: data → skårer → tap → gradienter → oppdatering. Legg så til evaluering uten oppdatering.

</details>


In [ ]:
own_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))
own_train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
own_test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
own_optimizer = torch.optim.SGD(own_model.parameters(), lr=0.1)
own_loss_fn = nn.CrossEntropyLoss()
own_model.train()
loss_sum = 0.0
for images, labels in own_train_loader:
    own_optimizer.zero_grad()
    scores = own_model(images)
    loss = own_loss_fn(scores, labels)
    loss.backward()
    own_optimizer.step()
    loss_sum += loss.item() * len(images)
print(f"Treningstap: {loss_sum / len(train_data):.4f}")

own_model.eval()
num_correct = 0
with torch.no_grad():
    for images, labels in own_test_loader:
        predictions = own_model(images).argmax(dim=1)
        num_correct += (predictions == labels).sum().item()
print(f"Testnøyaktighet: {num_correct / len(test_data):.1%}")


### Refleksjon

Forklar hele kjeden fra en gruppe bilder til oppdaterte parametere.
Ta med tensorformene, tapsfunksjonens rolle og forskjellen mellom gradientberegning
og parameteroppdatering. Forklar også hvorfor evaluering er en separat løkke.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

En gruppe `[B, 1, 28, 28]` flates til `[B, 784]` og blir til `[B, 64]`, ReLU beholder formen, og siste lag gir rå skårer `[B, 10]`.
Kryssentropitapet sammenligner skårene med etiketter `[B]` og gir ett tapsmål.
Vi nullstiller gamle gradienter, og `backward()` beregner nye gradienter for
modellens parametere. `step()` oppdaterer parametrene i retning av lavere tap.
Evalueringen bruker egne testbilder uten gradientberegning eller oppdateringer,
slik at vi måler prediksjoner på bilder modellen ikke lærte fra.


## Valgfri ekstraoppgave

Lag en forvekslingsmatrise for den ferdigtrente 128-modellen: rader er riktige
sifre, kolonner er predikerte sifre. Vis den med `imshow`, fargeskala og norske
aksetekster. Undersøk hvilke forvekslinger som er vanligst. Bruk eksisterende
prediksjoner; ikke tren eller velg nye modeller ut fra testresultatet.


In [ ]:
matrix = torch.zeros(10, 10, dtype=torch.int64)
for target, predicted in zip(results[128]["labels"], results[128]["predictions"]):
    matrix[target, predicted] += 1
fig, ax = plt.subplots(figsize=(7, 6))
display = ax.imshow(matrix.numpy(), cmap="Blues")
fig.colorbar(display, ax=ax, label="Antall bilder")
ax.set(xlabel="Predikert siffer", ylabel="Riktig siffer", title="Forvekslingsmatrise")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
plt.show()


## Valgfri fordypning: 32 eller 128 skjulte nevroner?

Sammenlign de to forhåndsvalgte størrelsene. Du har allerede 128-resultatet.
Tren en ny 32-modell med samme frø, egen lastergenerator med frø 42, batchstørrelse,
læringsrate og antall epoker. Lagre i de samme ordbøkene med nøkkel `32`.
Skriv ut resultater og tegn treningskurvene i samme figur.

Samme frø gir ikke identiske startvekter i modeller med ulike former.
Ett forsøk kan ikke bevise at flere nevroner alltid er bedre. Ikke bruk
sammenligningen til videre tuning på testsettet.


In [ ]:
seed_everything(42)
model32 = create_model(32)
loader32 = DataLoader(train_data, batch_size=64, shuffle=True,
                      generator=torch.Generator().manual_seed(42))
start = perf_counter()
histories[32] = train_model(model32, loader32)
training_times[32] = perf_counter() - start
results[32] = evaluate_model(model32, test_loader)
models[32] = model32
fig, ax = plt.subplots()
for size in [32, 128]:
    print(size, "sluttap:", histories[size][-1],
          "testnøyaktighet:", results[size]["accuracy"],
          "sekunder:", training_times[size])
    ax.plot(range(1, 6), histories[size], marker="o", label=f"{size} nevroner")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="To skjulte lagstørrelser")
ax.legend()
plt.show()


### Refleksjon

Oppgi begge modellresultatene fra din kjøring. Hvilken praktisk forskjell ser du?
Kan vi konkludere med at den ene størrelsen alltid er bedre?


**Mitt svar:**

Tabellen og figuren viser målt forskjell i tap, nøyaktighet og tid. Hvis
nøyaktigheten er lik, har den større modellen ikke gitt bedre resultat etter
dette målet i denne kjøringen. Ett frø og 1 000 testbilder gir begrenset
støtte for generelle konklusjoner. Treningstid varierer også mellom maskiner.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Repetisjon:** [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/). Gå tilbake til forklaringen av skjulte lag etter at du har trent ditt eget nettverk. Knytt illustrasjonene til vektene, aktiveringene og formene i modellen din.
- **Kodeoppslag:** [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html). Se spesielt hvordan `Linear` og `ReLU` settes sammen til et nettverk med skjulte lag.
- **Videre forståelse:** [Google Machine Learning Crash Course – Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting). Forklarer overtilpasning og generalisering. Bruk figurene til å diskutere hvorfor fallende treningstap alene ikke er nok til å vurdere modellen.
